In [ ]:

class RealTimeJumpDetector:

    def __init__(self,
                 window_size=30,
                 signature_depth=3,
                 threshold_quantile=0.98):

        self.W = window_size
        self.depth = signature_depth
        self.threshold_quantile = threshold_quantile

        # rolling window
        self.buffer = deque(maxlen=self.W)

        # history for variance norm
        self.signature_history = []

        self.threshold = None

    # ----------------------------
    # Compute path signature
    # ----------------------------

    def compute_signature(self, segment):

        segment = torch.tensor(segment, dtype=torch.float32)
        segment = segment.view(1, self.W, 2)

        sig = signatory.signature(segment, self.depth)

        return sig.detach().numpy()[0]

    # ----------------------------
    # Variance norm
    # ----------------------------

    def variance_norm(self, x, X):

        X = np.array(X)

        mu = X.mean(axis=0)

        cov = np.cov(X.T)

        inv_cov = pinv(cov)

        d = (x-mu).T @ inv_cov @ (x-mu)

        return d

    # ----------------------------
    # Update with new price
    # ----------------------------

    def update(self, data):

        self.buffer.append(data)

        if len(self.buffer) < self.W:
            return None

        segment = np.array(self.buffer)

        sig = self.compute_signature(segment)

        # warmup
        if len(self.signature_history) < 50:

            self.signature_history.append(sig)
            return None

        vnorm = self.variance_norm(sig, self.signature_history)

        self.signature_history.append(sig)

        # compute threshold dynamically
        vnorms = []

        for s in self.signature_history:

            d = self.variance_norm(s, self.signature_history)
            vnorms.append(d)

        self.threshold = np.quantile(vnorms, self.threshold_quantile)

        jump = vnorm > self.threshold

        return {

            "jump": jump,
            "vnorm": vnorm,
            "threshold": self.threshold
        }
        
    def classify_jump(self, sig):

        price_move = sig[0]
        volume_move = sig[1]

        if price_move > 0 and volume_move > 0:
            return "upward_jump"

        elif price_move < 0 and volume_move > 0:
            return "downward_jump"

        elif abs(price_move) > 0 and abs(volume_move) < 0.1:
            return "make_market"

        return None

In [ ]:
import torch
import numpy as np
from collections import deque
import signatory


class SignatureKernelJumpDetector:

    def __init__(self,
                 window_size=30,
                 signature_depth=3,
                 threshold_quantile=0.98):

        self.W = window_size
        self.depth = signature_depth
        self.threshold_quantile = threshold_quantile

        self.buffer = deque(maxlen=self.W)

        self.signature_history = []

        self.threshold = None

    # --------------------------------
    # compute signature
    # --------------------------------

    def compute_signature(self, segment):

        segment = torch.tensor(segment, dtype=torch.float32)
        segment = segment.view(1, self.W, segment.shape[1])

        sig = signatory.signature(segment, self.depth)

        return sig.detach().numpy()[0]

    # --------------------------------
    # signature kernel
    # --------------------------------

    def kernel_score(self, sig, history):

        history = np.array(history)

        # linear signature kernel
        similarities = history @ sig

        score = np.mean(similarities)

        return score

    # --------------------------------
    # update
    # --------------------------------

    def update(self, data):

        self.buffer.append(data)

        if len(self.buffer) < self.W:
            return None

        segment = np.array(self.buffer)

        sig = self.compute_signature(segment)

        if len(self.signature_history) < 50:
            self.signature_history.append(sig)
            return None

        score = self.kernel_score(sig, self.signature_history)

        self.signature_history.append(sig)

        scores = []

        for s in self.signature_history:
            scores.append(self.kernel_score(s, self.signature_history))

        self.threshold = np.quantile(scores, self.threshold_quantile)

        jump = score < self.threshold

        return {
            "jump": jump,
            "score": score,
            "threshold": self.threshold
        }

In [ ]:
plot_data = []
for idx, snap_idx, vnorm, threshold in mah_jump_list:
    plot_data.append({
        "snap_idx": snap_idx,
        "vnorm": vnorm,
        "threshold": threshold,
        "bid" : df_market[(df_market['snap_idx'] == snap_idx) & (df_market['side'] == 'Bid')]['price'].values[0],
        "ask" : df_market[(df_market['snap_idx'] == snap_idx) & (df_market['side'] == 'Ask')]['price'].values[0]
    })
    
df_jumps = pd.DataFrame(plot_data)

for idx, snap_idx, score, threshold in ker_jump_list:
    plot_data.append({
        "snap_idx": snap_idx,
        "score": score,
        "threshold": threshold,
        "bid" : df_market[(df_market['snap_idx'] == snap_idx) & (df_market['side'] == 'Bid')]['price'].values[0],
        "ask" : df_market[(df_market['snap_idx'] == snap_idx) & (df_market['side'] == 'Ask')]['price'].values[0]
    })
    
df_ker_jumps = pd.DataFrame(plot_data)
    

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot bid and ask on primary y-axis
ax1.plot(df_jumps['snap_idx'], df_jumps['bid'], label='Bid Price', color='blue')
ax1.plot(df_jumps['snap_idx'], df_jumps['ask'], label='Ask Price', color='red')
ax1.set_xlabel('Snap Index')
ax1.set_ylabel('Price Value')
ax1.legend(loc='upper left')

# Create secondary y-axis for threshold
ax2 = ax1.twinx()
ax2.plot(df_jumps['snap_idx'], df_jumps['threshold'], label='Threshold', linestyle='--', color='green')
ax2.plot(df_jumps['snap_idx'], df_jumps['vnorm'], label='Variance Norm', linestyle=':', color='orange')

ax2.plot(df_ker_jumps['snap_idx'], df_ker_jumps['threshold'], label='Kernel Threshold', linestyle='--', color='purple')
ax2.plot(df_ker_jumps['snap_idx'], df_ker_jumps['score'], label='Kernel Score', linestyle=':', color='brown')

ax2.plot
ax2.set_ylabel('Threshold Value')
ax2.legend(loc='upper right')

plt.title('Bid/Ask Prices and Threshold over Time')
plt.show()